# Lead-Lag Correlation (Group 03 · Step 01) — SYNTHETIC DEMO

> **What this notebook does** — the same `03-01` correlation pipeline as the real one, but on **synthetic** data. It reads the summaries produced by `demo/02-01-etl-summary.ipynb` from `/kaggle/working`, applies the quality filters + **ETF/ETN/ETV/ETS type exclusion** (using the synthetic `tickers.parquet`), runs the lead-lag search on a random 1000-ticker sample, and writes the discovered pairs **locally** to `/kaggle/working`.

## Job

| | |
|---|---|
| **Reads (input)** | `/kaggle/working/summary/minute_summary` + `daily_volume` (from demo 02-01) + synthetic `summary/tickers/tickers.parquet` from the mounted datasets |
| **Writes (output)** | `/kaggle/working/strategies/correlation/{timestamp}/data.parquet` |
| **Libraries** | `pyspark` (local) |
| **Pipeline role** | stage 03 — discovers lead-lag pairs on the synthetic universe |

> **Run order:** `demo/02-01` first (produces the summaries), then this notebook.


In [ ]:
# ============================================================================
# Setup -- secrets, S3 client, DuckDB S3 helper, Spark session
# ============================================================================

!pip install -q duckdb --upgrade

import sys
import os
import shutil
import json

# Copy the autotrade package files from the Kaggle dataset into a proper
# package directory, then add it to sys.path.
# Locate the autotrade-package dataset wherever it mounts (mount path varies).
import glob as _glob
_pkg_cands = sorted(_glob.glob("/kaggle/input/**/creds.json", recursive=True))
if _pkg_cands:
    src_pkg = os.path.dirname(_pkg_cands[0])
else:
    src_pkg = '/kaggle/input/autotrade-package'
dst_pkg = '/kaggle/working/autotrade'
if os.path.isdir(src_pkg) and not os.path.isdir(dst_pkg):
    os.makedirs(dst_pkg, exist_ok=True)
    for fname in os.listdir(src_pkg):
        if fname.endswith('.py'):
            shutil.copy2(os.path.join(src_pkg, fname), os.path.join(dst_pkg, fname))
    print(f"Copied autotrade package from dataset to working directory")

sys.path.insert(0, '/kaggle/working')

import time
from io import BytesIO

import numpy as np
import pandas as pd
import boto3
import duckdb
import math
from datetime import datetime, timedelta

from IPython.display import display, HTML

from pyspark.sql import SparkSession, DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import *

from autotrade.config import load_config
from autotrade.storage import s3_client, duckdb_s3_connect as _duckdb_s3_connect, spark_session
from autotrade.correlation import compute_bh_fdr, compute_market_returns_decimal, analyze_pairs, run_lagged_stock_search
from autotrade.features import prepare_features
from autotrade.filters import apply_all_filters, apply_quality_filters, limit_tickers, get_stats

# Load credentials from autotrade-package dataset (instead of UserSecretsClient)
with open(os.path.join(src_pkg, 'creds.json')) as f:
    creds = json.load(f)

cfg = load_config()
cfg.aws_access_key_id = creds["aws_access_key_id"]
cfg.aws_secret_access_key = creds["aws_secret_access_key"]
cfg.massive_api_key = creds["massive_api_key"]

REGION_NAME = cfg.aws_region
MY_BUCKET   = cfg.s3_bucket

region_name = REGION_NAME
my_bucket   = MY_BUCKET

SPARK_TMP   = cfg.spark_tmp
os.makedirs(SPARK_TMP, exist_ok=True)

s3 = s3_client(cfg)


def duckdb_s3_connect():
    return _duckdb_s3_connect(cfg)


spark = spark_session(cfg)

# Suppress verbose Spark logging and query plans
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.debug.maxToStringFields", "100")

print("Setup complete")


In [ ]:
# ============================================================================
# Analysis parameters
# ============================================================================

import random

# --- Randomized lead-lag parameters (new values each run) --------------------
# Synthetic sector lead-lags are 2..10 days (generator: lag = 2 + sector % 9),
# so randomize within that window to guarantee aligned pairs are found.
lag_days           = random.randint(2, 10)     # prediction horizon (days)
lookback_days      = random.randint(1, 30)      # rolling-average window (days)
persistence_window = random.randint(5, 30)      # rolling-correlation window (days)
print(f"Randomized: lag_days={lag_days}, lookback_days={lookback_days}, persistence_window={persistence_window}")

# --- Data window (full summary range: 2024-02-01 -> 2026-08-28) -------------
start_date = "2024-02-01"        # inclusive start
end_date   = "2026-01-01"        # inclusive end

# --- Ticker universe ---------------------------------------------------------
max_tickers = 1000               # random sample size, applied AFTER quality filters

# --- Lead-lag analysis -------------------------------------------------------
add_significance   = True        # add significance score/rank columns
min_significance   = 0.1         # significance_score threshold (relaxed to surface more pairs)
adjust_returns     = False        # market-adjust returns before correlation
fdr_q              = 0.10        # Benjamini-Hochberg FDR threshold

# --- Quality filters (applied in order; ticker limit is LAST) ----------------
min_observations   = 150         # min rows per ticker
min_coverage_ratio = 0.7         # calendar coverage ratio
max_gap_days       = 60          # regular-trading max gap (days); 60 tolerates the 2026-01-01->02-23 collection gap
min_avg_volume     = 100         # liquidity floor (avg daily share volume)
max_daily_vol      = 100.0        # volatility ceiling; NOTE: filter uses std of PRICE level, so synthetic prices up to ~$250 need a high ceiling


# --- Ticker type exclusions (ETF, ETN, ETV, ETS) --------------------------------
# Tickers metadata comes from the mounted synthetic datasets (mount path varies).
import glob as _glob
EXCLUDED_TICKER_TYPES = ["ETF", "ETN", "ETV", "ETS"]
TICKER_TYPES_PATHS = sorted(_glob.glob("/kaggle/input/**/summary/tickers/tickers.parquet", recursive=True))
if not TICKER_TYPES_PATHS:
    raise SystemExit("synthetic tickers.parquet not found in /kaggle/input - attach "
                     "dsptlp/synthetic-market-data (+ -02) and run demo/02-01 first")
print("TICKER_TYPES_PATHS =", TICKER_TYPES_PATHS)

# --- Data source: summaries produced by demo/02-01 -------------------------
# Preferred: the derived synthetic-market-data-summary dataset (separate session).
# Fallback: /kaggle/working (if demo/02-01 ran in THIS same session).
_sum_cands = sorted(_glob.glob("/kaggle/input/**/minute_summary/data.parquet", recursive=True))
_vol_cands = sorted(_glob.glob("/kaggle/input/**/daily_volume/data.parquet", recursive=True))
if not _sum_cands and os.path.isdir("/kaggle/working/summary/minute_summary/data.parquet"):
    _sum_cands = ["/kaggle/working/summary/minute_summary/data.parquet"]
if not _vol_cands and os.path.isdir("/kaggle/working/summary/daily_volume/data.parquet"):
    _vol_cands = ["/kaggle/working/summary/daily_volume/data.parquet"]
if not _sum_cands or not _vol_cands:
    raise SystemExit("summaries not found - run demo/02-01-etl-summary.ipynb first "
                     "(it publishes to dsptlp/synthetic-market-data-summary)")
DATA_SOURCE_LOCAL = _sum_cands[0]
DATA_SOURCE_VOLUME = _vol_cands[0]
print("DATA_SOURCE_LOCAL =", DATA_SOURCE_LOCAL)
print("DATA_SOURCE_VOLUME =", DATA_SOURCE_VOLUME)

# --- High-quality pair selection (SQL query thresholds) ----------------------
min_correlation   = 0.30          # |correlation| floor (relaxed)
min_sign_fraction = 0.00         # |sign_fraction| floor (dataset max ~0.13 with market-adjusted returns)
recent_days       = 20           # rolling-corr recency window (days)
result_limit      = 500          # max pairs per run


# (ticker types resolved above via glob from the mounted synthetic datasets)

# --- Data source -------------------------------------------------------------
# (DATA_SOURCE_LOCAL/VOLUME resolved above from the summary dataset or /kaggle/working)

## Load & prepare data


In [ ]:

# ============================================================================
# CELL 25: Load Raw Data
# ============================================================================

print("Loading data from Parquet...")
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")
df_raw = spark.read.parquet(DATA_SOURCE_LOCAL)

# Check schema
df_raw.printSchema()


In [ ]:
# ============================================================================
# Load daily close prices AND real daily volume from 02-01 summary
# ============================================================================

# Filter to daily close (last minute bar of each day)
# (df_raw was already loaded + schema-printed in the cell above)
df_raw = df_raw.filter(F.col("rn_desc").isin(1))

# Load real daily volume from 02-01 summary
vol = spark.read.parquet(DATA_SOURCE_VOLUME)

df = (
    df_raw
    .select(
        F.col("symbol").alias("ticker"),
        F.col("close").alias("price"),
        F.col("trade_date").alias("date"),
    )
    .join(
        vol.select(
            F.col("symbol").alias("ticker"),
            F.col("trade_date").alias("date"),
            F.col("volume").alias("volume"),   # REAL daily share volume
            F.col("trades").alias("trades"),   # daily trade count (for reference)
        ),
        ["ticker", "date"],
        "left"
    )
    .cache()  # Cache to avoid re-reading Parquet + redoing join
)

# Materialize cache once
total_obs = df.count()
print(f"{total_obs:,} daily observations with real volume")



In [ ]:
# ============================================================================
# One-time deterministic filters + cache (runs ONCE before the loop)
# ============================================================================

# ============================================================================
# Ticker type exclusion (ETF, ETN, ETV, ETS) - applied BEFORE quality filters
# ============================================================================

# Read ticker types from the Kaggle dataset
tickers_df = (
    spark.read.parquet(*TICKER_TYPES_PATHS)
    .select("ticker", "type")
    .dropDuplicates(["ticker"])
)

# Identify excluded tickers
excluded_tickers = (
    tickers_df.filter(F.col("type").isin(EXCLUDED_TICKER_TYPES))
    .select("ticker").distinct()
    .collect()
)
excluded_set = {r["ticker"] for r in excluded_tickers}
print(f"Excluding {len(excluded_set):,} tickers of types {EXCLUDED_TICKER_TYPES}")

# Anti-join to remove excluded tickers from the universe
excluded_df = (
    F.broadcast(
        tickers_df.filter(F.col("type").isin(EXCLUDED_TICKER_TYPES))
        .select(F.col("ticker").alias("excluded_ticker"))
    )
)
df = (
    df.join(excluded_df, df.ticker == F.col("excluded_ticker"), "left")
    .filter(F.col("excluded_ticker").isNull())
    .drop("excluded_ticker")
)

before_rows, before_tickers = get_stats(df, "After ticker type exclusion")
print(f"  Rows: {before_rows:,}, Tickers: {before_tickers:,}")


print("=" * 70)
print("APPLYING DATA QUALITY FILTERS (once, cached)")
print("=" * 70)

df_filtered = apply_quality_filters(
    df,
    start_date=start_date,
    end_date=end_date,
    min_rows=min_observations,
    min_coverage_ratio=min_coverage_ratio,
    max_gap_days=max_gap_days,
    min_avg_volume=min_avg_volume,
    max_daily_vol=max_daily_vol,
    track=True,   # prints the per-filter impact table ONCE
).cache()

# For display: one-time sample for the "FINAL CLEANED DATASET" panel
#df_clean = limit_tickers(df_filtered, max_tickers=max_tickers)

initial_rows, initial_tickers = get_stats(df, "Initial")
final_rows, final_tickers = get_stats(df_filtered, "Final")

print("-" * 85)
print(f"{'TOTAL REMOVED':<30} {'-':>15} {'-':>10} {initial_rows - final_rows:>15,} {initial_tickers - final_tickers:>15,}")
print(f"{'RETENTION RATE':<30} {final_rows/initial_rows*100:>14.1f}% {final_tickers/initial_tickers*100:>9.1f}% {'-':>15} {'-':>15}")

print("-" * 85)
print("✅ FINAL CLEANED DATASET:")
print("=" * 70)
print(f"Observations:  {final_rows:>12,} ({final_rows/initial_rows*100:.1f}% of original)")
print(f"Tickers:       {final_tickers:>12,} ({final_tickers/initial_tickers*100:.1f}% of original)")
print("=" * 70)

print("Cleaned data sample:")
df_filtered.limit(5).toPandas().to_html(escape=False)

# Clear summary of filtered dataset
final_rows = df_filtered.count()
final_tickers = df_filtered.select("ticker").distinct().count()
initial_rows = df.count()
initial_tickers = df.select("ticker").distinct().count()

print("\n" + "=" * 70)
print("FINAL CLEANED DATASET")
print("=" * 70)
print(f"  Observations:  {final_rows:,} ({final_rows/initial_rows*100:.1f}% of original)")
print(f"  Tickers:       {final_tickers:,} ({final_tickers/initial_tickers*100:.1f}% of original)")
print("=" * 70)


## Run analysis


In [ ]:
# ============================================================================
# Single-pass analysis (optimized)
# ============================================================================

import time
from datetime import datetime, timedelta
from io import BytesIO
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

DEBUG = True

sector_df = None
market_df = None

pair_count = 0
file_key = None
elapsed = 0

# ---------------------------------------------------------------------------
# Market returns
# ---------------------------------------------------------------------------

try:
    market_df = compute_market_returns_decimal(df_filtered)

    if DEBUG:
        market_count = market_df.count()
        print(f"   [OK] Computed market returns for {market_count:,} dates")
    else:
        print("   [OK] Computed market returns")

except Exception as e:
    print(f"   [WARN] Could not compute market returns: {e}")
    print("   [INFO] Proceeding without market adjustment")
    market_df = None


print("=" * 70)
print("STARTING SINGLE-PASS ANALYSIS")
print("=" * 70)

start_time = time.time()

try:

    # ========================================================================
    # 1. Random ticker sample
    # ========================================================================

    print("\n[1/4] Applying fresh random sample...")

    df_iter = limit_tickers(
        df_filtered,
        max_tickers=max_tickers,
        seed=None
    )

    if DEBUG:
        ticker_count = (
            df_iter
            .select("ticker")
            .distinct()
            .count()
        )

        print(f"   OK {ticker_count:,} tickers selected")
    else:
        print(f"   OK ticker sample prepared (max {max_tickers:,})")


    # ========================================================================
    # 2. Lead-lag analysis
    # ========================================================================

    print("\n[2/4] Running lead-lag analysis...")

    results = run_lagged_stock_search(
        df_iter,
        lag_days=lag_days,
        lookback_days=lookback_days,
        persistence_window_days=persistence_window,
        add_significance=add_significance,
        min_significance=min_significance,
        fdr_q=fdr_q,
        sector_df=sector_df,
        market_df=market_df,
        adjust_returns=adjust_returns,
        debug=DEBUG,
    )

    print("   OK Lead-lag analysis complete")


    # ========================================================================
    # 3. Query high-quality pairs
    # ========================================================================

    print("\n[3/4] Querying high-quality pairs...")

    # Column names
    CORRELATION_COL         = "correlation"
    SIGN_FRACTION_COL       = "sign_fraction"
    AVG_FOLLOWER_CHANGE_COL = "avg_follower_" + str(lag_days) + "d_price_change"
    AVG_PRE_RETURN_COL      = "avg_pre_" + str(lag_days) + "d_return"
    SIGNIFICANCE_COL        = "significance_score"
    ROLLING_CORR_COL        = "rolling_corr"
    LEADER_DATE_COL         = "leader_date"
    LEADER_COL              = "leader"
    FOLLOWER_COL            = "follower"

    # Parameters
    MIN_SIGNIFICANCE  = min_significance
    MIN_CORRELATION   = min_correlation
    MIN_SIGN_FRACTION = min_sign_fraction
    RECENT_DAYS       = recent_days
    RESULT_LIMIT      = result_limit

    # Register once for SQL
    results.createOrReplaceTempView("stock_pairs")


    # ========================================================================
    # Optional diagnostics
    #
    # These can be expensive because each action triggers Spark work.
    # Set DEBUG=True when investigating a run.
    # ========================================================================

    if DEBUG:

        print("\n[debug] results-level stats...")

        _d = results.agg(
            F.count("*").alias("total_rows"),
            F.count("correlation").alias("corr_nonnull"),
            F.min("correlation").alias("corr_min"),
            F.max("correlation").alias("corr_max"),
            F.count("sign_fraction").alias("signfrac_nonnull"),
            F.min("sign_fraction").alias("signfrac_min"),
            F.max("sign_fraction").alias("signfrac_max"),
        ).first()

        print(
            f"   total_rows={_d.total_rows:,} "
            f"corr_nonnull={_d.corr_nonnull:,} "
            f"corr_range=[{_d.corr_min}, {_d.corr_max}] "
            f"signfrac_nonnull={_d.signfrac_nonnull:,} "
            f"signfrac_range=[{_d.signfrac_min}, {_d.signfrac_max}]"
        )

        _chain = spark.sql(f"""
            SELECT
                COUNT(*) AS total,

                SUM(
                    CASE
                        WHEN {SIGNIFICANCE_COL} >= {MIN_SIGNIFICANCE}
                        THEN 1 ELSE 0
                    END
                ) AS sig_pass,

                SUM(
                    CASE
                        WHEN {SIGNIFICANCE_COL} >= {MIN_SIGNIFICANCE}
                         AND ABS({CORRELATION_COL}) >= {MIN_CORRELATION}
                        THEN 1 ELSE 0
                    END
                ) AS corr_pass,

                SUM(
                    CASE
                        WHEN {SIGNIFICANCE_COL} >= {MIN_SIGNIFICANCE}
                         AND ABS({CORRELATION_COL}) >= {MIN_CORRELATION}
                         AND ABS({SIGN_FRACTION_COL}) >= {MIN_SIGN_FRACTION}
                        THEN 1 ELSE 0
                    END
                ) AS all_pass,

                SUM(
                    CASE
                        WHEN {CORRELATION_COL} IS NULL
                        THEN 1 ELSE 0
                    END
                ) AS corr_null

            FROM stock_pairs
        """).toPandas()

        print("   [debug] drop chain:")
        print(_chain.to_string(index=False))

        _top = (
            results
            .filter(F.col(CORRELATION_COL).isNotNull())
            .orderBy(F.desc(F.abs(F.col(CORRELATION_COL))))
            .limit(5)
            .select(
                LEADER_COL,
                FOLLOWER_COL,
                CORRELATION_COL,
                SIGN_FRACTION_COL
            )
            .toPandas()
        )

        print("   [debug] top 5 rows by |correlation|:")
        print(_top.to_string(index=False))


    # ========================================================================
    # IMPORTANT FIX
    #
    # Calculate MAX(leader_date) separately.
    #
    # Do NOT put:
    #
    #     SELECT MAX(leader_date) FROM stock_pairs
    #
    # inside AVG().
    #
    # Spark rejects that scalar subquery as a nondeterministic expression
    # inside the aggregate.
    # ========================================================================

    max_leader_date = (
        results
        .select(
            F.max(F.col(LEADER_DATE_COL)).alias("max_leader_date")
        )
        .first()["max_leader_date"]
    )

    if max_leader_date is None:
        raise ValueError("No leader_date values found in results")

    recent_cutoff_date = (
        max_leader_date - timedelta(days=RECENT_DAYS)
    )

    print(f"   Max leader date:     {max_leader_date}")
    print(f"   Recent cutoff date: {recent_cutoff_date}")


    # ========================================================================
    # Final query
    # ========================================================================

    query = f"""
        SELECT *
        FROM (
            SELECT

                {LEADER_COL},
                {FOLLOWER_COL},
                {CORRELATION_COL},
                {SIGN_FRACTION_COL},
                {AVG_FOLLOWER_CHANGE_COL},
                {AVG_PRE_RETURN_COL},
                {SIGNIFICANCE_COL},

                ABS({CORRELATION_COL}) AS abs_correlation,

                ABS({AVG_FOLLOWER_CHANGE_COL}) AS abs_expected_move,

                COUNT(*) AS num_observations,

                AVG(
                    CASE
                        WHEN {LEADER_DATE_COL} >=
                            TIMESTAMP('{recent_cutoff_date}')
                        THEN {ROLLING_CORR_COL}
                    END
                ) AS recent_{RECENT_DAYS}d_correlation,

                MAX({LEADER_DATE_COL}) AS last_observation_date,

                ROW_NUMBER() OVER (
                    PARTITION BY
                        {LEADER_COL},
                        {FOLLOWER_COL}

                    ORDER BY
                        ABS({CORRELATION_COL})
                        * ABS({SIGN_FRACTION_COL}) DESC
                ) AS rn

            FROM stock_pairs

            WHERE
                {SIGNIFICANCE_COL} >= {MIN_SIGNIFICANCE}

            GROUP BY
                {LEADER_COL},
                {FOLLOWER_COL},
                {CORRELATION_COL},
                {SIGN_FRACTION_COL},
                {AVG_FOLLOWER_CHANGE_COL},
                {AVG_PRE_RETURN_COL},
                {SIGNIFICANCE_COL}

            HAVING
                ABS({CORRELATION_COL}) >= {MIN_CORRELATION}
                AND ABS({SIGN_FRACTION_COL}) >= {MIN_SIGN_FRACTION}
        )

        WHERE rn = 1

        ORDER BY
            ABS({CORRELATION_COL}) DESC,
            ABS({SIGN_FRACTION_COL}) DESC,
            ABS({AVG_FOLLOWER_CHANGE_COL}) DESC

        LIMIT {RESULT_LIMIT}
    """


    # ========================================================================
    # Execute final query
    #
    # Don't call count() first. The result is immediately going to Pandas,
    # so count() would cause an unnecessary second Spark action.
    # ========================================================================

    high_quality_pairs = spark.sql(query)

    high_quality_pairs_pd = high_quality_pairs.toPandas()

    pair_count = len(high_quality_pairs_pd)

    print(
        f"   OK {pair_count:,} unique high-quality pairs "
        f"(final result)"
    )


    # ========================================================================
    # 4. Upload results to S3
    # ========================================================================

    print("\n[4/4] Uploading results to S3...")

    run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    high_quality_pairs_pd["run_timestamp"] = run_timestamp
    high_quality_pairs_pd["iteration"] = 1
    high_quality_pairs_pd["lag_days"] = lag_days
    high_quality_pairs_pd["lookback_days"] = lookback_days
    high_quality_pairs_pd["persistence_window"] = persistence_window

    # Convert datetime columns before Parquet serialization
    for col in high_quality_pairs_pd.columns:
        if pd.api.types.is_datetime64_any_dtype(
            high_quality_pairs_pd[col]
        ):
            high_quality_pairs_pd[col] = (
                high_quality_pairs_pd[col].astype(str)
            )

    out_path = (
        f"/kaggle/working/strategies/correlation/"
        f"{run_timestamp}/data.parquet"
    )

    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    high_quality_pairs_pd.to_parquet(
        out_path,
        index=False,
        engine="pyarrow",
        coerce_timestamps="ms",
        allow_truncated_timestamps=True
    )

    elapsed = time.time() - start_time

    print(
        f"   OK Wrote to "
        f"{out_path}"
    )

    print(
        f"\nOK Analysis complete in "
        f"{timedelta(seconds=int(elapsed))}"
    )


except Exception as e:

    elapsed = time.time() - start_time

    print(f"\nERROR Analysis failed: {e}")

    import traceback
    traceback.print_exc()

    # Preserve the original exception instead of producing a
    # secondary NameError in the summary.
    raise


# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

print(f"  Unique pairs found:  {pair_count:,}")
print(f"  Local location:      {out_path}")
print(f"  Runtime:             {timedelta(seconds=int(elapsed))}")

print(
    f"  Parameters:          "
    f"lag={lag_days}d, "
    f"lookback={lookback_days}d, "
    f"persistence={persistence_window}d"
)

print(
    f"  Significance:        "
    f"min={min_significance}, "
    f"corr>={min_correlation}, "
    f"sign>={min_sign_fraction}"
)

print("=" * 70)